## Setup

In [66]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor


import mlflow
import dagshub
import mlflow.sklearn



In [67]:
# MLflow
dagshub.init(
    repo_owner='IzaKakhniashvili',
    repo_name='ML-assignment1-HousePrices',
    mlflow=True
)


Initialized MLflow to track repo "IzaKakhniashvili/ML-assignment1-HousePrices"

Repository IzaKakhniashvili/ML-assignment1-HousePrices initialized!

In [68]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [69]:
train.shape, test.shape

((1460, 81), (1459, 80))

In [70]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

In [71]:
train["SalePrice"].describe()

count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50%      163000.000000
75%      214000.000000
max      755000.000000
Name: SalePrice, dtype: float64

In [72]:
train.isnull().sum().sort_values(ascending=False).head(20)

PoolQC          1453
MiscFeature     1406
Alley           1369
Fence           1179
MasVnrType       872
FireplaceQu      690
LotFrontage      259
GarageYrBlt       81
GarageCond        81
GarageType        81
GarageFinish      81
GarageQual        81
BsmtFinType2      38
BsmtExposure      38
BsmtQual          37
BsmtCond          37
BsmtFinType1      37
MasVnrArea         8
Electrical         1
Id                 0
dtype: int64

## Baseline

In [115]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)


In [116]:
# mode/median ით დავჰენდლეთ ყველაზე მარტივად
for col in X.columns:
    if X[col].dtype == "object":
        X[col] = X[col].fillna(X[col].mode()[0])
    else:
        X[col] = X[col].fillna(X[col].median())

In [117]:
X = pd.get_dummies(X)
X

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,60,65.0,8450,7,5,2003,2003,196.0,706,...,False,False,False,True,False,False,False,False,True,False
1,2,20,80.0,9600,6,8,1976,1976,0.0,978,...,False,False,False,True,False,False,False,False,True,False
2,3,60,68.0,11250,7,5,2001,2002,162.0,486,...,False,False,False,True,False,False,False,False,True,False
3,4,70,60.0,9550,7,5,1915,1970,0.0,216,...,False,False,False,True,True,False,False,False,False,False
4,5,60,84.0,14260,8,5,2000,2000,350.0,655,...,False,False,False,True,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,1456,60,62.0,7917,6,5,1999,2000,0.0,0,...,False,False,False,True,False,False,False,False,True,False
1456,1457,20,85.0,13175,6,6,1978,1988,119.0,790,...,False,False,False,True,False,False,False,False,True,False
1457,1458,70,66.0,9042,7,9,1941,2006,0.0,275,...,False,False,False,True,False,False,False,False,True,False
1458,1459,20,68.0,9717,5,6,1950,1996,0.0,49,...,False,False,False,True,False,False,False,False,True,False


In [118]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [119]:
model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [120]:
preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse

np.float64(29558.54872308715)

In [121]:
r2 = r2_score(y_val, preds)
r2

0.886092484333137

In [122]:
mae = mean_absolute_error(y_val, preds)
mae

18262.222434916726

In [123]:
import mlflow

with mlflow.start_run():

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "baseline")

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 00:12:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 00:12:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run inquisitive-jay-698 at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/a4260105c4c547cba6609d730ceca63f
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


KeyboardInterrupt: 

## Experiment 1

#### Data Cleaning

In [124]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [125]:
X_tr = X_train.copy()
X_v = X_val.copy()

In [126]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())

In [127]:
cat_cols = X_tr.select_dtypes(include=['object']).columns
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Engineering

In [128]:
for df in [X_tr, X_v]:
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']

df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,TotalSF,HouseAge
892,893,20,RL,70.0,8414,Pave,None,Reg,Lvl,AllPub,...,None,MnPrv,None,0,2,2006,WD,Normal,2127,43
1105,1106,60,RL,98.0,12256,Pave,None,IR1,Lvl,AllPub,...,None,None,None,0,4,2010,WD,Normal,4085,16
413,414,30,RM,56.0,8960,Pave,Grvl,Reg,Lvl,AllPub,...,None,None,None,0,3,2010,WD,Normal,2036,83
522,523,50,RM,50.0,5000,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,10,2006,WD,Normal,2668,59
1036,1037,20,RL,89.0,12898,Pave,None,IR1,HLS,AllPub,...,None,None,None,0,9,2009,WD,Normal,3240,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
479,480,30,RM,50.0,5925,Pave,None,Reg,Bnk,AllPub,...,None,MnPrv,None,0,3,2007,WD,Alloca,2038,70
1361,1362,20,RL,124.0,16158,Pave,None,IR1,Low,AllPub,...,None,None,None,0,6,2009,WD,Normal,3060,4
802,803,60,RL,63.0,8199,Pave,None,Reg,Lvl,AllPub,...,None,None,None,0,10,2008,WD,Normal,2184,3
651,652,70,RL,60.0,9084,Pave,None,Reg,Lvl,AllPub,...,None,MnPrv,None,0,10,2009,WD,Normal,2265,69


In [129]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])

combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

#### Feature Selection

In [130]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()

threshold = 0.4
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
relevant_cols


['OverallQual',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'TotalBsmtSF',
 '1stFlrSF',
 'GrLivArea',
 'FullBath',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'TotalSF',
 'HouseAge',
 'ExterQual_Ex',
 'ExterQual_Gd',
 'ExterQual_TA',
 'Foundation_PConc',
 'BsmtQual_Ex',
 'BsmtQual_TA',
 'BsmtFinType1_GLQ',
 'HeatingQC_Ex',
 'KitchenQual_Ex',
 'KitchenQual_TA',
 'FireplaceQu_None',
 'GarageFinish_Fin',
 'GarageFinish_Unf',
 'SalePrice']

In [131]:
relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

#### Train

In [132]:
model_a = LinearRegression()
model_a.fit(X_tr_final, y_train)
preds = model_a.predict(X_v_final)

In [133]:
rmse = np.sqrt(mean_squared_error(y_val, preds))
rmse

np.float64(34766.144680111094)

In [134]:
r2 = r2_score(y_val, preds)
r2

0.8424206763480444

In [136]:
mae = mean_absolute_error(y_val, preds)
mae

22149.141288559178

In [137]:
with mlflow.start_run(run_name="Exp_A"):

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "Experiment A")
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 00:18:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 00:18:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 00:19:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 5
Created version '5' of model 'house_price_model'.


🏃 View run Exp_A at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/a8283618ddac44918d02486a8678c2ec
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Experiment 2

#### Data Cleaning

In [138]:
X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [141]:
X_tr = X_train.copy()
X_v = X_val.copy()

In [140]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())

X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Engineering

In [142]:
for df in [X_tr, X_v]:
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df.fillna(0, inplace=True)

In [143]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [144]:
y_train_log = np.log1p(y_train)

#### Feature Selection

In [146]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()
threshold = 0.2
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')
relevant_cols

['LotFrontage',
 'LotArea',
 'OverallQual',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'GrLivArea',
 'BsmtFullBath',
 'FullBath',
 'HalfBath',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'TotalSF',
 'TotalBath',
 'HouseAge',
 'MSZoning_RL',
 'MSZoning_RM',
 'LotShape_Reg',
 'Neighborhood_NoRidge',
 'Neighborhood_NridgHt',
 'Neighborhood_StoneBr',
 'HouseStyle_2Story',
 'RoofStyle_Gable',
 'RoofStyle_Hip',
 'Exterior1st_VinylSd',
 'Exterior2nd_VinylSd',
 'MasVnrType_0',
 'MasVnrType_Stone',
 'ExterQual_Ex',
 'ExterQual_Gd',
 'ExterQual_TA',
 'Foundation_CBlock',
 'Foundation_PConc',
 'BsmtQual_Ex',
 'BsmtQual_Gd',
 'BsmtQual_TA',
 'BsmtExposure_Gd',
 'BsmtExposure_No',
 'BsmtFinType1_GLQ',
 'HeatingQC_Ex',
 'HeatingQC_TA',
 'CentralAir_N',
 'CentralAir_Y',
 'Electrical_SBrkr',
 'KitchenQual_Ex',
 'KitchenQual_Gd',
 'KitchenQual_TA',
 'FireplaceQu_0

In [147]:
X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [148]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [149]:
model = LinearRegression()
model.fit(X_tr_scaled, y_train_log)

log_preds = model.predict(X_v_scaled)
final_preds = np.expm1(log_preds)

In [150]:
rmse = np.sqrt(mean_squared_error(y_val, final_preds))
rmse

np.float64(27263.946555368682)

In [151]:
r2 = r2_score(y_val, preds)
r2

0.8424206763480444

In [152]:
mae = mean_absolute_error(y_val, preds)
mae

22149.141288559178

In [153]:
with mlflow.start_run(run_name="Exp_B"):

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "Experiment B")
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 00:46:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 00:46:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 00:46:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 6
Created version '6' of model 'house_price_model'.


🏃 View run Exp_B at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/8bf83d13f1a548d0baf23787f68860e6
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Experiment 3

#### Data Cleaning

In [174]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [175]:
X_tr = X_train.copy()
X_v = X_val.copy()

In [176]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Engineering

In [177]:
for df in [X_tr, X_v]:
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df.fillna(0, inplace=True)

In [178]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [179]:
numeric_feats = X_tr_enc.dtypes[X_tr_enc.dtypes != "object"].index
skewed_feats = X_tr_enc[numeric_feats].apply(lambda x: x.skew()).sort_values(ascending=False)

# 0.75-ზე მეტი ასიმეტრია
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index

In [180]:
for feat in high_skew:
        X_tr_enc[feat] = np.log1p(X_tr_enc[feat])
        X_v_enc[feat] = np.log1p(X_v_enc[feat])

In [181]:
y_train_log = np.log1p(y_train)

#### Feature Selection

In [182]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()
threshold = 0.2
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

relevant_cols

['LotFrontage',
 'LotArea',
 'OverallQual',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 'GrLivArea',
 'BsmtFullBath',
 'FullBath',
 'HalfBath',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'TotalSF',
 'TotalBath',
 'HouseAge',
 'MSZoning_RL',
 'MSZoning_RM',
 'LotShape_Reg',
 'Neighborhood_NoRidge',
 'Neighborhood_NridgHt',
 'Neighborhood_StoneBr',
 'HouseStyle_2Story',
 'RoofStyle_Gable',
 'RoofStyle_Hip',
 'Exterior1st_VinylSd',
 'Exterior2nd_VinylSd',
 'MasVnrType_None',
 'MasVnrType_Stone',
 'ExterQual_Ex',
 'ExterQual_Gd',
 'ExterQual_TA',
 'Foundation_CBlock',
 'Foundation_PConc',
 'BsmtQual_Ex',
 'BsmtQual_Gd',
 'BsmtQual_TA',
 'BsmtExposure_Gd',
 'BsmtExposure_No',
 'BsmtFinType1_GLQ',
 'HeatingQC_Ex',
 'HeatingQC_TA',
 'CentralAir_N',
 'CentralAir_Y',
 'Electrical_SBrkr',
 'KitchenQual_Ex',
 'KitchenQual_Gd',
 'KitchenQual_TA',
 'FireplaceQu_Ex',
 'FireplaceQu_Gd',
 '

In [183]:
X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [184]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [185]:
model = LinearRegression()
model.fit(X_tr_scaled, y_train_log)

log_preds = model.predict(X_v_scaled)
final_preds = np.expm1(log_preds)

In [186]:
rmse = np.sqrt(mean_squared_error(y_val, final_preds))
rmse

np.float64(27780.36317444251)

In [187]:
r2 = r2_score(y_val, preds)
r2

0.8424206763480444

In [188]:
mae = mean_absolute_error(y_val, preds)
mae

22149.141288559178

In [189]:
with mlflow.start_run(run_name="Exp_B"):

    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "Experiment 3")
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("skew_threshold", 0.75)
    mlflow.log_param("final_features", len(relevant_cols))
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 01:12:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:12:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 01:12:25 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 7
Created version '7' of model 'house_price_model'.


🏃 View run Exp_B at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/38c1b3d3ec4146e7897e7b12f2cdb7fa
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## Experiment 4

#### Data Cleaning

In [190]:
y = train["SalePrice"]
X = train.drop("SalePrice", axis=1)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

X_tr = X_train.copy()
X_v = X_val.copy()
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Engineering

In [191]:
for df in [X_tr, X_v]:
        df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
        df['TotalBath'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
        df['HouseAge'] = df['YrSold'] - df['YearBuilt']
        df.fillna(0, inplace=True)

In [192]:
X_tr['is_train'] = 1
X_v['is_train'] = 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [193]:
numeric_feats = X_tr_enc.dtypes[X_tr_enc.dtypes != "object"].index
skewed_feats = X_tr_enc[numeric_feats].apply(lambda x: x.skew()).sort_values(ascending=False)
high_skew = skewed_feats[abs(skewed_feats) > 0.75].index

for feat in high_skew:
    X_tr_enc[feat] = np.log1p(X_tr_enc[feat])
    X_v_enc[feat] = np.log1p(X_v_enc[feat])

#### Feature Selection

In [197]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
corr_matrix = temp_df.corr()
threshold = 0.2
relevant_cols = corr_matrix['SalePrice'][abs(corr_matrix['SalePrice']) > threshold].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [198]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [199]:
y_train_log = np.log1p(y_train)

In [201]:
from sklearn.linear_model import LassoCV

lasso_model = LassoCV(alphas=np.logspace(-4, -1, 30), cv=5, max_iter=10000, random_state=42)
lasso_model.fit(X_tr_scaled, y_train_log)

,"eps eps: float, default=1e-3Length of the path. ``eps=1e-3`` means that``alpha_min / alpha_max = 1e-3``.",0.001
,"n_alphas n_alphas: int, default=100Number of alphas along the regularization path... deprecated:: 1.7 `n_alphas` was deprecated in 1.7 and will be removed in 1.9. Use `alphas` instead.",'deprecated'
,"alphas alphas: array-like or int, default=NoneValues of alphas to test along the regularization path.If int, `alphas` values are generated automatically.If array-like, list of alpha values to use... versionchanged:: 1.7 `alphas` accepts an integer value which removes the need to pass `n_alphas`... deprecated:: 1.7 `alphas=None` was deprecated in 1.7 and will be removed in 1.9, at which point the default value will be set to 100.","array([0.0001..., 0.1 ])"
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto false, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: 'auto', bool or array-like of shape (n_features, n_features), default='auto'Whether to use a precomputed Gram matrix to speed upcalculations. If set to ``'auto'`` let us decide. The Grammatrix can also be passed as argument.",'auto'
,"max_iter max_iter: int, default=1000The maximum number of iterations.",10000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``.",0.0001
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"cv cv: int, cross-validation generator or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- int, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For int/None inputs, :class:`~sklearn.model_selection.KFold` is used.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: bool or int, default=FalseAmount of verbosity.",False
,"n_jobs n_jobs: int, default=NoneNumber of CPUs to use during the cross validation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [202]:
log_preds = lasso_model.predict(X_v_scaled)
final_preds = np.expm1(log_preds)

In [203]:
rmse = np.sqrt(mean_squared_error(y_val, final_preds))
rmse

np.float64(28080.52011404558)

In [204]:
mae = mean_absolute_error(y_val, final_preds)
mae

17285.18403633688

In [205]:
r2 = r2_score(y_val, final_preds)
r2

0.8971992078855165

In [206]:
used_features = np.sum(lasso_model.coef_ != 0)
used_features

np.int64(45)

In [208]:
with mlflow.start_run(run_name="Exp_4_Lasso"):
    mlflow.log_param("model_type", "LassoCV")
    mlflow.log_param("best_alpha", lasso_model.alpha_)
    mlflow.log_param("features_selected", used_features)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("stage", "Experiment 4")
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 01:49:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 01:49:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 01:49:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 8
Created version '8' of model 'house_price_model'.


🏃 View run Exp_4_Lasso at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/033911be5ba241d7826d1b9de38ab5c0
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## 5. Remove Outliers

#### Data Cleaning

In [215]:
df_d = train.copy()
df_d = df_d.drop(df_d[(df_d['GrLivArea'] > 4000) & (df_d['SalePrice'] < 300000)].index)

In [216]:
y_d = df_d["SalePrice"]
X_d = df_d.drop("SalePrice", axis=1)
X_train_d, X_val_d, y_train_d, y_val_d = train_test_split(X_d, y_d, test_size=0.2, random_state=42)

In [217]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns
X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Engineering

In [218]:
for df in [X_tr, X_v]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df.fillna(0, inplace=True)

In [219]:
X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)
X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)


#### Feature Selection

In [221]:
temp_df = pd.concat([X_tr_enc, y_train_d], axis=1)
relevant_cols = temp_df.corr()['SalePrice'][abs(temp_df.corr()['SalePrice']) > 0.2].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [222]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [224]:
y_train_log = np.log1p(y_train_d)
lr_model_final = LinearRegression()

lr_model_final.fit(X_tr_scaled, y_train_log)
preds_e = np.expm1(lr_model_final.predict(X_v_scaled))

In [225]:
rmse = np.sqrt(mean_squared_error(y_val_d, final_preds))
rmse

np.float64(22827.78749168159)

In [228]:
r2 = r2_score(y_val_d, final_preds)
r2

0.9056601008788424

In [229]:
mae = mean_absolute_error(y_val_d, final_preds)
mae

16305.758241376188

In [230]:
with mlflow.start_run(run_name="Exp_5_Outliers"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 5")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 02:25:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 02:25:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 02:25:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 9
Created version '9' of model 'house_price_model'.


🏃 View run Exp_4_Lasso at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/940824e38d144871ab8c7e1688911a95
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## 6

#### Data Cleaning

In [253]:
df_f = train.copy()
#ვშლით გიგანტურ, მაგრამ იაფ სახლებს
df_f = df_f.drop(df_f[(df_f['GrLivArea'] > 4000) & (df_f['SalePrice'] < 300000)].index)

y_f = df_f["SalePrice"]
X_f = df_f.drop("SalePrice", axis=1)

X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

In [254]:
X_tr = X_train_f.copy()
X_v = X_val_f.copy()

In [255]:
missing_pct = X_tr.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.8].index.tolist()
if 'FireplaceQu' in cols_to_drop: cols_to_drop.remove('FireplaceQu')

In [256]:
X_tr = X_tr.drop(columns=cols_to_drop)
X_v = X_v.drop(columns=cols_to_drop)

In [257]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
existing_qual_cols = [c for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond'] if c in X_tr.columns]

In [258]:
for col in existing_qual_cols:
    X_tr[col] = X_tr[col].fillna('None').map(qual_map)
    X_v[col] = X_v[col].fillna('None').map(qual_map)


#### Feature Engineering

In [259]:
for df in [X_tr, X_v]:
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['Quality_x_Size'] = df['OverallQual'] * df['GrLivArea']
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemod'] = df['YrSold'] - df['YearRemodAdd']

In [260]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

#### Feature Selection

In [261]:
X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.concat([X_tr, X_v])
combined_encoded = pd.get_dummies(combined, columns=cat_cols)

X_tr_enc = combined_encoded[combined_encoded['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined_encoded[combined_encoded['is_train'] == 0].drop('is_train', axis=1)

In [262]:
temp_df = pd.concat([X_tr_enc, y_train_f], axis=1)
relevant_cols = temp_df.corr()['SalePrice'][abs(temp_df.corr()['SalePrice']) > 0.2].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]


In [263]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [265]:
y_train_log = np.log1p(y_train_f)
model = LinearRegression()
model.fit(X_tr_scaled, y_train_log)
final_preds = np.expm1(model.predict(X_v_scaled))

In [266]:
rmse = np.sqrt(mean_squared_error(y_val_f, final_preds))
rmse

np.float64(23555.48396751046)

In [267]:
r2 = r2_score(y_val_f, final_preds)
r2

0.8995495633761422

In [280]:
mae = mean_absolute_error(y_val_f, final_preds)
mae

15278.959273600985

In [269]:
with mlflow.start_run(run_name="Exp_6"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 6")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 17:21:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 17:21:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 17:21:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 10
Created version '10' of model 'house_price_model'.


🏃 View run Exp_6 at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/b4cdfab63eba4d0fa3a348420d66a489
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## 7

#### Data Cleaning

In [233]:
df = train.copy()
df = df.drop(df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300000)].index)

y = df["SalePrice"]
X = df.drop("SalePrice", axis=1)


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [270]:
X_tr = X_train.copy()
X_v = X_val.copy()


In [271]:
missing_pct = X_tr.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.89].index.tolist()
if 'FireplaceQu' in cols_to_drop: cols_to_drop.remove('FireplaceQu') # სასარგებლოა

X_tr = X_tr.drop(columns=cols_to_drop)
X_v = X_v.drop(columns=cols_to_drop)

In [272]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols = [c for c in ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu'] if c in X_tr.columns]

for col in qual_cols:
    X_tr[col] = X_tr[col].fillna('None').map(qual_map)
    X_v[col] = X_v[col].fillna('None').map(qual_map)

#### Feature Engineering

In [273]:
for df_temp in [X_tr, X_v]:
    df_temp['TotalSF'] = df_temp['TotalBsmtSF'] + df_temp['1stFlrSF'] + df_temp['2ndFlrSF']
    df_temp['TotalBath'] = df_temp['FullBath'] + (0.5 * df_temp['HalfBath']) + df_temp['BsmtFullBath']
    df_temp['HouseAge'] = df_temp['YrSold'] - df_temp['YearBuilt']
    df_temp['IsRemodeled'] = (df_temp['YearRemodAdd'] != df_temp['YearBuilt']).astype(int)

In [274]:
skew_cols = ['LotArea', 'GrLivArea', '1stFlrSF', 'TotalBsmtSF']
for col in skew_cols:
    X_tr[col] = np.log1p(X_tr[col])
    X_v[col] = np.log1p(X_v[col])

num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")

X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.get_dummies(pd.concat([X_tr, X_v]), columns=cat_cols)
X_tr_enc = combined[combined['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined[combined['is_train'] == 0].drop('is_train', axis=1)

#### Feature Selection

In [275]:
temp_df = pd.concat([X_tr_enc, y_train], axis=1)
relevant_cols = temp_df.corr()['SalePrice'][abs(temp_df.corr()['SalePrice']) > 0.15].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

In [276]:
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [277]:
y_train_log = np.log1p(y_train)
model = RidgeCV(alphas=np.logspace(-2, 2, 20), cv=5)
model.fit(X_tr_scaled, y_train_log)
final_preds = np.expm1(model.predict(X_v_scaled))

In [278]:
rmse = np.sqrt(mean_squared_error(y_val, final_preds))
rmse

np.float64(21690.147830402606)

In [279]:
r2 = r2_score(y_val, final_preds)
r2

0.9148287954168752

In [281]:
mae = mean_absolute_error(y_val_f, final_preds)
mae

15278.959273600985

In [282]:
with mlflow.start_run(run_name="Exp_7"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 7")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", threshold)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 17:27:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 17:27:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 17:28:05 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 11
Created version '11' of model 'house_price_model'.


🏃 View run Exp_7 at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/b66dc4f03a3a4a7f9ff08eb2071d49b3
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0


## 8

#### Data Cleaning

In [283]:
df_h = train.copy()
df_h = df_h.drop(df_h[(df_h['GrLivArea'] > 4000) & (df_h['SalePrice'] < 300000)].index)

y_h = df_h["SalePrice"]
X_h = df_h.drop("SalePrice", axis=1)

X_train_h, X_val_h, y_train_h, y_val_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

In [284]:
X_tr = X_train_h.copy()
X_v = X_val_h.copy()

In [285]:
temp_df = pd.concat([X_tr, y_train_h], axis=1)
neigh_map = temp_df.groupby('Neighborhood')['SalePrice'].median().sort_values()

In [286]:
bins = pd.qcut(neigh_map, 3, labels=[1, 2, 3]).to_dict()

In [287]:
X_tr['Neigh_Rank'] = X_tr['Neighborhood'].map(bins).astype(float)
X_v['Neigh_Rank'] = X_v['Neighborhood'].map(bins).astype(float).fillna(2.0)

In [288]:
binary_features = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea']
for col in binary_features:
    if col in X_tr.columns:
        X_tr[f'Has_{col}'] = (X_tr[col] > 0).astype(int)
        X_v[f'Has_{col}'] = (X_v[col] > 0).astype(int)

In [289]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_cols = ['ExterQual', 'KitchenQual', 'BsmtQual', 'HeatingQC', 'FireplaceQu']
for col in qual_cols:
    if col in X_tr.columns:
        X_tr[col] = X_tr[col].fillna('None').map(qual_map)
        X_v[col] = X_v[col].fillna('None').map(qual_map)

#### Feature Engineering

In [290]:
for df_temp in [X_tr, X_v]:
    df_temp['TotalSF'] = df_temp['TotalBsmtSF'] + df_temp['1stFlrSF'] + df_temp['2ndFlrSF']
    df_temp['Total_Qual'] = df_temp['OverallQual'] + df_temp['OverallCond']
    df_temp['GrLivArea'] = np.log1p(df_temp['GrLivArea'])
    df_temp['LotArea'] = np.log1p(df_temp['LotArea'])

In [291]:
num_cols = X_tr.select_dtypes(include=[np.number]).columns
cat_cols = X_tr.select_dtypes(include=['object']).columns

X_tr[num_cols] = X_tr[num_cols].fillna(X_tr[num_cols].median())
X_v[num_cols] = X_v[num_cols].fillna(X_tr[num_cols].median())
X_tr[cat_cols] = X_tr[cat_cols].fillna("None")
X_v[cat_cols] = X_v[cat_cols].fillna("None")


In [292]:
X_tr['is_train'], X_v['is_train'] = 1, 0
combined = pd.get_dummies(pd.concat([X_tr, X_v]), columns=cat_cols)
X_tr_enc = combined[combined['is_train'] == 1].drop('is_train', axis=1)
X_v_enc = combined[combined['is_train'] == 0].drop('is_train', axis=1)

#### Feature Selection

In [293]:
temp_corr = pd.concat([X_tr_enc, y_train_h], axis=1)
relevant_cols = temp_corr.corr()['SalePrice'][abs(temp_corr.corr()['SalePrice']) > 0.12].index.tolist()
if 'SalePrice' in relevant_cols: relevant_cols.remove('SalePrice')

X_tr_final = X_tr_enc[relevant_cols]
X_v_final = X_v_enc[relevant_cols]

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_final)
X_v_scaled = scaler.transform(X_v_final)

In [294]:
y_train_log = np.log1p(y_train_h)
model = RidgeCV(alphas=np.logspace(-1, 2, 30), cv=5)
model.fit(X_tr_scaled, y_train_log)

final_preds = np.expm1(model.predict(X_v_scaled))

In [295]:
rmse = np.sqrt(mean_squared_error(y_val_h, final_preds))
rmse

np.float64(20855.937104731413)

In [296]:
r2 = r2_score(y_val_h, final_preds)
r2

0.921254239340018

In [297]:
mae = mean_absolute_error(y_val_f, final_preds)
mae

14768.330581471459

In [298]:
with mlflow.start_run(run_name="Exp_8"):
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("stage", "Experiment 8")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2_score", r2)
    mlflow.log_param("corr_threshold", 0.12)
    mlflow.log_param("final_features", len(relevant_cols))


    mlflow.sklearn.log_model(
        model,
        "model",
        registered_model_name="house_price_model"
    )

2026/04/13 17:34:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 17:35:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house_price_model' already exists. Creating a new version of this model...
2026/04/13 17:35:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house_price_model, version 12
Created version '12' of model 'house_price_model'.


🏃 View run Exp_8 at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0/runs/a4c60538c43a4adcb2acf59f4787fc30
🧪 View experiment at: https://dagshub.com/IzaKakhniashvili/ML-assignment1-HousePrices.mlflow/#/experiments/0
